# Defining Boundary Conditions

Written for dolfinx v0.7.2, python 3.10.6

With the mesh completed, the next step is to define the values of the fields at the boundary of the mesh.

# Libraries / Packages

In [1]:
from mpi4py import MPI # This is the communicator (or message passing interface) that allows python to work in parallel

# Create the MPI communicator (This has to happen before any other code is run.)
comm = MPI.COMM_WORLD
print(f"size = {comm.Get_size()}")
print(f"rank = {comm.Get_rank()}")

import numpy as np # For working with arrays
import dolfinx # Dolfinx is FEniCSx's computation backend / user interface

size = 1
rank = 0


# Variables & Parameters

In [2]:
# TODO again these params are just guesses right now

V_c = 0 # V, electric potential at surface of cathode
V_a = 3000 # V, electric potential at surface of anode

t_trig = 5e-3 # s, time at which the trigger stops firing
def V_trig(t): # V, electric potential at surface of trigger as a function of time
    '''
    Args:
        - t: time (in seconds)
    Returns:
        - electric potential at surface of trigger at time t
    '''
    if t < 0:
        raise ValueError("Time must be greater than or equal to zero")
    elif t < t_trig: # While triggering
        return 3000
    else: # Not triggering
        return V_c


# Input Pre-Defined Mesh

This is collimator_tetra.msh file that was defined in gmsh_collimator.ipynb. GMSH is separate to FEniCSx, and FEniCSx requires the mesh to be in a xmdf format. The file is converted using meshio. 

In [3]:
# Read mesh from the XDMF file (it automatically reads the linked .h5 file)
with dolfinx.io.XDMFFile(comm, "meshes/collimator_tetra.xdmf", "r") as xdmf: 
    mesh = xdmf.read_mesh(name="Grid") 
    print("mesh read") 
    cell_tags = xdmf.read_meshtags(mesh, name="Grid") 

mesh read


In [ ]:

# Get the tags for each surface - this is from the gmsh_collimator jupyter notebook. 
surface_tags = {
    "collimator_surface": 53,
    "anode_surface": 54,
    "cathode_surface": 55,
    "trigger_surface": 56,
    "magnets_surface": 57,
}

volume_tags = {
    "collimator": 49,
    "anode": 50,
    "cathode": 51,
    "trigger": 52,
}


# Define boundary conditions

There are certain boundary conditions that apply to the mesh, which are implemented before simulation can begin.

The majority of these are Dirichlet boundary conditions, which take the form:

$$ \phi(\vec{r}) |_{s} = f(\vec{r}) $$

Where $f(\vec{r})$ is some known function, and $\phi(\vec{r}) |_{s}$ is the electric potential evaluated at the surface S. For example, the cathode, anode, and trigger are held at a known potential difference.

Typically, electromagnetic problems set the potential to zero at an infinite distance from the charges. However, this is not implemented here for multiple reasons:

1)  FEniCSx is not able to model infinite distances. 
    When modelling infinite values computationally, the typical strategy is to choose a suitably large number instead of infinity (i.e., set the potential to zero at a distance of 10 metres from the thruster and collimator.). However, the size of the mesh must be very small to accurately capture ion-ion interactions within the plasma. Therefore it would require significant computing power to model a sufficiently large space. 

2)  The potential at an infinite distance is unlikely to be zero in real life, due to the influence of the Earth's magnetic field (during both vacuum chamber testing and in low-Earth orbit) as well as the ionosphere, other satellites, and solar weather in orbit.

Instead, a background magnetic field is modelled. Since the simulation will be experimentally verified in a vacuum chamber on Earth, the background magnetic field will be approximately 1 Gauss. In low-Earth orbit, this value would be lower. 

To define the boundary conditions, a function space $V$ is created on the imported mesh. 


In [6]:
# This process is described in the FEniCSx tutorial documentation here: https://jsdokken.com/dolfinx-tutorial/chapter3/subdomains.html#convert-msh-files-to-xdmf-using-meshio

# Define a function space on the mesh.
# Note: 
# - Lagrangian is a smooth function L described here: https://defelement.org/elements/lagrange.html
# - Degree 1 means points are defined at the vertices, whereas (for example) degree 2 has points at the midpoints as well. 
V = dolfinx.fem.functionspace(mesh, ("Lagrange", 1))

# Get the dimension of the entity to apply the BC on (Dirichlet conditions are applied on the boundary, 
# so will have dimension 1 lower than the problem dimension. The boundary is a 2d surface in this case.)
fdim = mesh.topology.dim - 1


# Locate degrees-of-freedom belonging to mesh entities topologically
cathode_dofs = dolfinx.fem.locate_dofs_topological(V, fdim, cell_tags.find(surface_tags["cathode_surface"]))
anode_dofs = dolfinx.fem.locate_dofs_topological(V, fdim, cell_tags.find(surface_tags["anode_surface"]))

# Apply a dirichlet BC for the different values
bcs = [dolfinx.fem.dirichletbc(dolfinx.default_scalar_type(V_c), cathode_dofs, V), 
       dolfinx.fem.dirichletbc(dolfinx.default_scalar_type(V_a), anode_dofs, V)
       ]